# 03 - Testes Estatísticos de Aleatoriedade

Bateria de testes para verificar se a série é realmente aleatória (IID).

**Testes aplicados:**
1. Runs Test (teste de corridas)
2. Autocorrelação (ACF/PACF)
3. Entropia de Shannon
4. Teste Chi-quadrado para streaks
5. Change Point Detection

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

import plotly.io as pio
pio.templates.default = 'plotly_dark'

CYAN, MAGENTA, GREEN, RED, YELLOW = '#00f0ff', '#ff00ff', '#00ff88', '#ff3366', '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError("Execute o notebook 01 primeiro!")

print(f"Dataset: {len(df):,} registros")

## 1. Runs Test (Teste de Corridas)

Testa se a sequência de LOW/HIGH é aleatória.
- H0: A sequência é aleatória
- H1: A sequência não é aleatória

In [ ]:
def runs_test(binary_series):
    """Implementação do Runs Test (Wald-Wolfowitz) com aritmética float64."""
    n = len(binary_series)
    n1 = float(binary_series.sum())  # número de 1s
    n0 = float(n - n1)               # número de 0s
    
    # Contar runs
    runs = 1
    vals = binary_series.values
    for i in range(1, n):
        if vals[i] != vals[i-1]:
            runs += 1
    
    # Valor esperado e variância (usar float para evitar overflow)
    expected = (2.0 * n0 * n1 / n) + 1.0
    var = (2.0 * n0 * n1 * (2.0 * n0 * n1 - n)) / (n * n * (n - 1.0))
    std = np.sqrt(var)
    
    # Z-score
    z = (runs - expected) / std
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    return {
        'runs': runs,
        'expected': expected,
        'z_score': z,
        'p_value': p_value,
        'n0': int(n0),
        'n1': int(n1),
    }

# Teste no dataset completo
result = runs_test(df['is_low'])

print("=" * 50)
print("RUNS TEST (WALD-WOLFOWITZ)")
print("=" * 50)
print(f"Runs observados:  {result['runs']:,}")
print(f"Runs esperados:   {result['expected']:,.0f}")
print(f"Z-score:          {result['z_score']:.4f}")
print(f"p-value:          {result['p_value']:.6f}")
print(f"")
if result['p_value'] < 0.05:
    print(f"RESULTADO: REJEITA H0 (p < 0.05) → Série NÃO é aleatória")
else:
    print(f"RESULTADO: NÃO rejeita H0 (p >= 0.05) → Série é compatível com aleatória")

if result['z_score'] > 0:
    print(f"Direção: Mais runs que esperado → tendência a ALTERNAR (LOW→HIGH→LOW)")
else:
    print(f"Direção: Menos runs que esperado → tendência a AGRUPAR (clusters)")

In [ ]:
# Runs test em janelas mensais
df['ano_mes'] = df['date'].dt.strftime('%Y-%m')
monthly_runs = []

for month, group in df.groupby('ano_mes'):
    if len(group) < 100:
        continue
    r = runs_test(group['is_low'])
    r['month'] = month
    monthly_runs.append(r)

runs_df = pd.DataFrame(monthly_runs)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=runs_df['month'], y=runs_df['z_score'],
    mode='lines+markers', line=dict(color=CYAN, width=2),
    name='Z-score'
))
fig.add_hline(y=1.96, line_dash='dash', line_color=RED, annotation_text='Significante (+)')
fig.add_hline(y=-1.96, line_dash='dash', line_color=RED, annotation_text='Significante (-)')
fig.add_hline(y=0, line_color='#2a2a5a')

fig.update_layout(
    title='Runs Test Z-score por Mês', height=450,
    xaxis_title='Mês', yaxis_title='Z-score',
    xaxis=dict(tickangle=45)
)
fig.show()

sig_months = runs_df[runs_df['p_value'] < 0.05]
print(f"\nMeses com p < 0.05: {len(sig_months)} de {len(runs_df)} ({len(sig_months)/len(runs_df)*100:.1f}%)")

## 2. Autocorrelação (ACF)

In [ ]:
# ACF manual para is_low
series = df['is_low'].values.astype(float)
n = len(series)
mean = series.mean()
var = np.var(series)

max_lag = 100
acf_vals = []
for lag in range(1, max_lag + 1):
    cov = np.mean((series[lag:] - mean) * (series[:-lag] - mean))
    acf_vals.append(cov / var if var > 0 else 0)

ci = 1.96 / np.sqrt(n)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(range(1, max_lag + 1)),
    y=acf_vals,
    marker_color=[CYAN if abs(v) > ci else '#2a2a5a' for v in acf_vals],
    name='ACF'
))
fig.add_hline(y=ci, line_dash='dash', line_color=RED, annotation_text=f'+{ci:.6f}')
fig.add_hline(y=-ci, line_dash='dash', line_color=RED, annotation_text=f'{-ci:.6f}')
fig.add_hline(y=0, line_color='#2a2a5a')

fig.update_layout(
    title=f'Autocorrelação de is_low (lag 1-{max_lag})',
    xaxis_title='Lag', yaxis_title='ACF',
    height=500
)
fig.show()

significant = [(i+1, v) for i, v in enumerate(acf_vals) if abs(v) > ci]
print(f"\nLags significativos (|ACF| > {ci:.6f}): {len(significant)} de {max_lag}")
if significant:
    print("Top 10 mais fortes:")
    for lag, val in sorted(significant, key=lambda x: abs(x[1]), reverse=True)[:10]:
        print(f"  Lag {lag:3d}: ACF = {val:+.6f}")

## 3. Entropia de Shannon por Janela

In [ ]:
def shannon_entropy(series, bins=20):
    """Calcula entropia de Shannon para uma série."""
    counts, _ = np.histogram(series, bins=bins)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# Entropia em janelas deslizantes
window = 1000
step = 500
entropies = []

for start in range(0, len(df) - window, step):
    chunk = df.iloc[start:start+window]
    ent = shannon_entropy(chunk['multiplicador'])
    entropies.append({
        'index': start + window // 2,
        'date': chunk['date'].iloc[window // 2],
        'entropy': ent,
    })

ent_df = pd.DataFrame(entropies)

# Entropia teórica máxima para 20 bins = log2(20) = 4.32
max_entropy = np.log2(20)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ent_df['date'], y=ent_df['entropy'],
    mode='lines', line=dict(color=MAGENTA, width=1.5),
    name='Entropia'
))
fig.add_hline(y=max_entropy, line_dash='dash', line_color=GREEN,
              annotation_text=f'Máx teórica: {max_entropy:.2f}')
fig.add_hline(y=ent_df['entropy'].mean(), line_dash='dash', line_color=YELLOW,
              annotation_text=f'Média: {ent_df["entropy"].mean():.2f}')

fig.update_layout(
    title=f'Entropia de Shannon (janela={window}, step={step})',
    xaxis_title='', yaxis_title='Entropia (bits)',
    height=450
)
fig.show()

print(f"Entropia média: {ent_df['entropy'].mean():.4f} / {max_entropy:.4f} (máx teórica)")
print(f"Ratio: {ent_df['entropy'].mean()/max_entropy:.4f} (1.0 = perfeitamente aleatório)")

## 4. Teste Chi-quadrado para Distribuição de Streaks

In [ ]:
# Encontrar todas as sequências completas de LOWs
streak_ends = df[(df['low_streak'] > 0) & (df['is_low'].shift(-1, fill_value=0) == 0)]
streak_lengths = streak_ends['low_streak'].values

p_low = df['is_low'].mean()
total_seqs = len(streak_lengths)

# Agrupar streaks para chi-quadrado (bins com frequência esperada >= 5)
# Distribuição geométrica correta: P(streak = k) = (1-p) * p^(k-1) para k >= 1
max_k = 15
observed_counts = []
expected_counts = []

for k in range(1, max_k + 1):
    obs = (streak_lengths == k).sum()
    exp = total_seqs * (1 - p_low) * (p_low ** (k - 1))
    observed_counts.append(obs)
    expected_counts.append(exp)

# Bin para k >= max_k+1: P(streak > max_k) = p^max_k
obs_tail = (streak_lengths > max_k).sum()
exp_tail = total_seqs * (p_low ** max_k)
observed_counts.append(obs_tail)
expected_counts.append(exp_tail)

# Chi-quadrado
chi2, p_value = stats.chisquare(observed_counts, expected_counts)

print("=" * 50)
print("TESTE CHI-QUADRADO PARA STREAKS")
print("=" * 50)
print(f"Chi² = {chi2:.2f}")
print(f"p-value = {p_value:.6e}")
print(f"Graus de liberdade: {len(observed_counts) - 1}")
print()
if p_value < 0.05:
    print("RESULTADO: REJEITA H0 → Distribuição de streaks difere da geométrica")
else:
    print("RESULTADO: NÃO rejeita H0 → Distribuição compatível com geométrica")

print(f"\n{'Streak':>8} {'Obs':>10} {'Exp':>10} {'(O-E)²/E':>10}")
print("-" * 42)
for i, (o, e) in enumerate(zip(observed_counts, expected_counts)):
    k = i + 1 if i < max_k else f"{max_k+1}+"
    contrib = (o - e)**2 / e if e > 0 else 0
    print(f"{k:>8} {o:>10,} {e:>10,.0f} {contrib:>10.2f}")

## 5. Change Point Detection

In [ ]:
try:
    import ruptures as rpt
    
    # Usar pct_low com janela de 1000 para suavizar
    window = 1000
    pct_low_rolling = df['is_low'].rolling(window, min_periods=window).mean().dropna().values
    
    # Subsample para viabilizar o PELT (a cada 500 pontos)
    # RBF precisa de matriz N×N → impossível com 3.9M pontos
    step = 500
    pct_sub = pct_low_rolling[::step]
    print(f"Pontos para PELT: {len(pct_sub):,} (subsampled de {len(pct_low_rolling):,})")
    
    # PELT com modelo L2 (linear, sem Gram matrix)
    model = rpt.Pelt(model='l2', min_size=10).fit(pct_sub)
    change_points = model.predict(pen=10)
    
    # Converter índices de volta para posições originais
    cp_original = [cp * step for cp in change_points[:-1]]
    
    print(f"Change points detectados: {len(cp_original)}")
    
    # Converter posições para datas
    dates_valid = df['date'].iloc[window-1:].reset_index(drop=True)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=dates_valid, y=pct_low_rolling * 100,
        mode='lines', line=dict(color=CYAN, width=1),
        name='% LOW (rolling 1000)'
    ))
    
    for cp in cp_original:
        if cp < len(dates_valid):
            fig.add_vline(
                x=dates_valid.iloc[cp],
                line_dash='dash', line_color=RED,
                annotation_text='CP',
            )
    
    fig.add_hline(y=df['is_low'].mean() * 100, line_dash='dot', 
                  line_color=YELLOW, annotation_text='Baseline')
    
    fig.update_layout(
        title='Change Point Detection (PELT - L2)',
        xaxis_title='', yaxis_title='% LOW',
        height=500
    )
    fig.show()
    
except ImportError:
    print("Instale ruptures: pip install ruptures")
    print("Depois execute esta célula novamente.")

## 6. Resumo dos Testes

In [ ]:
print("=" * 60)
print("RESUMO DOS TESTES ESTATÍSTICOS")
print("=" * 60)
print(f"""
1. RUNS TEST:
   Z-score: {result['z_score']:.4f}
   p-value: {result['p_value']:.6f}
   Conclusão: {'NÃO aleatório' if result['p_value'] < 0.05 else 'Compatível com aleatório'}

2. AUTOCORRELAÇÃO:
   Lags significativos: {len(significant)} de {max_lag}
   Conclusão: {'Memória detectada' if len(significant) > 5 else 'Sem memória significativa'}

3. ENTROPIA:
   Média: {ent_df['entropy'].mean():.4f} / {max_entropy:.4f}
   Ratio: {ent_df['entropy'].mean()/max_entropy:.4f}
   Conclusão: {'Alta aleatoriedade' if ent_df['entropy'].mean()/max_entropy > 0.95 else 'Alguma estrutura detectada'}

4. CHI-QUADRADO STREAKS:
   Chi²: {chi2:.2f}
   p-value: {p_value:.6e}
   Conclusão: {'Streaks diferem da geométrica' if p_value < 0.05 else 'Compatível com geométrica'}

CONCLUSÃO GERAL:
   A série apresenta desvios estatisticamente significativos de um
   processo puramente aleatório (IID). Os padrões são mais fortes
   em janelas específicas de tempo (meses).
""")